In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import csv

# Mortgage Rate Enrichment

Merge the combined sold and listings datasets with FRED (national 30-year fixed mortgage rate from the St. Louis Federal Reserve). FRED is live economic data from a public API and, we performed a time-series join on a monthly key.

## Code

In [13]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

In [14]:
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = mortgage.groupby('year_month')['rate_30yr_fixed'].mean().reset_index()

In [15]:
sold = pd.read_csv('/Users/jessicakwandou/Documents/IDX Exchange/combined_sold.csv')
listings = pd.read_csv('/Users/jessicakwandou/Documents/IDX Exchange/listing_cleaned.csv')

sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')

listings['year_month'] = pd.to_datetime(listings['ListingContractDate']).dt.to_period('M')

/var/folders/13/041lbv8n5d1clk4p9mt3qr540000gp/T/ipykernel_28432/4273952468.py:1: DtypeWarning: Columns (2,7,45,78,79,80,81,82,83) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv('/Users/jessicakwandou/Documents/IDX Exchange/combined_sold.csv')


In [16]:
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
listings_with_rates = listings.merge(mortgage_monthly, on='year_month', how='left')

In [17]:
print(sold_with_rates['rate_30yr_fixed'].isnull().sum())
print(listings_with_rates['rate_30yr_fixed'].isnull().sum())

0
0


In [18]:
print(sold_with_rates[['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']].head())

    CloseDate year_month  ClosePrice  rate_30yr_fixed
0  2022-02-25    2022-02     95000.0           3.7625
1  2022-02-19    2022-02      1200.0           3.7625
2  2022-04-15    2022-04   1100000.0           4.9825
3  2022-01-04    2022-01   2499999.0           3.4450
4  2022-01-12    2022-01    640000.0           3.4450


In [19]:
sold_with_rates.to_csv("sold_with_rates.csv", index=False)
listings_with_rates.to_csv("listings_with_rates.csv", index=False)